# Exploiting Lie Group Equivariance & Symmetry on Unicycle Robots

We will show the benefits of disciplined body-frame canonicalization (taking advantage of group actions) and then left–right and front–back mirrors on unicycle robots. A feedforward MLP is enough for this platform. All data is real data from the Robotarium.


### Experimental Setup & Dataset

We have data from repeated runs in [Georgia Tech's Robotarium](https://www.robotarium.gatech.edu/) on robot 11 (because dynamics change subtly robot to robot) you can find the raw data in a google drive [here](https://drive.google.com/drive/folders/1clnovbZEz0lmSBDjoV2VMztKRxhU-Eog?usp=sharing).  For robot 11 it contains 11 total trajectories covering different speeds, angular velocities, forwards/backwards driving, spirals, which span the arena.

Data is recorded at 30hz, at each timestep we get the wheel encoder increment, and the ground truth Vicon (x,y,orientation), along with a variety of other sensor measurements we don't concern ourself with in this notebook.

For all training and evaluation the split is:

- **Test:** figure 8, diagonal lawnmower reverse, and random trajectory. These three collections are held out whole.
- **Training:** the first 75% of the other eight collections.
- **Validation:** the last 25% of those eight. This is the set used to choose checkpoints.

Each log is coarsened to windows of three ticks (about 0.1 s) before the cut, and a window never crosses it.


### The Unicycle Model Dynamics using Lie Groups (SE(2))

We employ the FLU (Forward Left Up) convention for the body frame. Given wheel encoder inputs ($v_l$ for the left wheel, $v_r$ for the right wheel), the unicycle's local (body-frame) velocities are:

$$
v_x^{\text{body}} = \frac{v_l + v_r}{2}
$$

$$
v_y^{\text{body}} = 0
$$

$$
\omega_\theta^{\text{body}} = \frac{v_r - v_l}{2}
$$

These are the components of a *twist* in the Lie algebra $\mathfrak{se}(2)$: an element of the tangent space at the identity of $SE(2)$. Each time step, we form an increment in the Lie algebra:

$$
\xi_k\ \equiv
\begin{pmatrix}
v_x^{\text{body}}\, \Delta t \\
0 \\
\omega_\theta^{\text{body}}\, \Delta t
\end{pmatrix}
\in \mathfrak{se}(2)
$$

To update the *global* pose, we apply this increment to the current pose via the $SE(2)$ group operation:

$$
g_{k+1} = g_k \cdot \exp_{SE(2)}(\xi_k)
$$

where $g_k \in SE(2)$ is the current pose (position and orientation as a rigid body transformation), and $\exp_{SE(2)}$ is the exponential map from the tangent space at the identity to the group.

In summary: at each step, the local (body-frame) displacement—defined in the tangent space at the identity—is mapped to the global coordinate frame by transforming via the group operation of $SE(2)$. This is the essence of compounding motions in the Lie group view.


### Where the Unicycle Model Breaks Down


We understand the dynamics of a unicycle robot quite well.  The equation there is a pretty good estimate, but let's see where they break down.


In [2]:
import sys
from pathlib import Path

import torch

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "src" / "learned_dynamics").exists() else HERE.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))

from unicycle.data import load_split
from unicycle.figures import (
    COLORS,
    UNICYCLE,
    plot_asymmetry,
    plot_loss,
    plot_response,
    plot_rollouts,
    plot_summary,
    plot_summary_paths,
    plot_test_rmse,
    plot_unicycle_rollouts,
)
from unicycle.models import (
    CanonicalModel,
    EvenOddModel,
    FrontBackSymmetricModel,
    NaiveModel,
    ResidualModel,
    SymmetricModel,
)
from unicycle.rollout import rollout_position_mse
from unicycle.train import load_checkpoint, save_checkpoint, train_open_loop

SEED = 0
EPOCHS = 100
LR = 1e-4
ROLLOUT_EVERY = 5
# Stop after this many epochs with no drop in the validation 5 s loss.
# The checkpoint is still the lowest validation position RMSE.
PATIENCE = 10
# Multiplier on the even-odd model's odd-branch size in the training loss.
# 0 drops the extra term. 2 is a heavy push toward the longitudinal map.
ODD_PENALTY = 0.0
# Unicycle path figures. Meters. None keeps the nominal 105 mm.
WHEEL_BASELINE = .105 # default

train_logs, val_logs, test_logs = load_split()
print(f"train {sum(log.n_poses for log in train_logs)} poses across {len(train_logs)} segments")
print(f"validation {sum(log.n_poses for log in val_logs)} poses across {len(val_logs)} segments")
print(f"test {sum(log.n_poses for log in test_logs)} poses across {len(test_logs)} segments")
for label, logs in (("train", train_logs), ("validation", val_logs), ("test", test_logs)):
    print(label, ", ".join(log.name for log in logs))


train 11675 poses across 8 segments
validation 3897 poses across 8 segments
test 6007 poses across 3 segments
train dynamic_slide (first 75%), lawnmower_diagonal (first 75%), lawnmower_horizontal (first 75%), lawnmower_horizontal_reverse (first 75%), lawnmower_vertical (first 75%), lawnmower_vertical_reverse (first 75%), spiral (first 75%), spiral_reverse (first 75%)
validation dynamic_slide (last 25%), lawnmower_diagonal (last 25%), lawnmower_horizontal (last 25%), lawnmower_horizontal_reverse (last 25%), lawnmower_vertical (last 25%), lawnmower_vertical_reverse (last 25%), spiral (last 25%), spiral_reverse (last 25%)
test figure8, lawnmower_diagonal_reverse, random_trajectory


In [ ]:
plot_unicycle_rollouts(val_logs, baseline=WHEEL_BASELINE, set_label="Validation")


In [ ]:
plot_unicycle_rollouts(test_logs, baseline=WHEEL_BASELINE, set_label="Test")


**Takeaways:** The unicycle model performs about as we'd expect, drifting over time, even after submillimeter measurements done with a caliper for the wheelbase length and the wheel radii.

Let's see how far a learned correction on top of this unicycle can go. That residual is the strongest baseline we can generate: the network only has to learn the slip.


### Physics Residual

As a baseline we train the same MLP to predict a residual body step on top of the wheel unicycle. The network sees $(\sigma, \delta)$, the same unicycle coordinates as canonical. The unicycle twist is applied outside it: rollout is $T \leftarrow T \circ Z \circ \operatorname{Exp}(r)$, with $Z = \operatorname{Exp}(v\Delta t,\, 0,\, \omega\Delta t)$ built from the ticks by the wheel geometry. A zero output is exactly the unicycle, so the network only has to learn the slip.

This is a strong prior, and the only model in which the radius and the wheel baseline are handed to the predictor. The learned regimes that follow never see that prior. The question is whether they still match or beat this baseline.


In [3]:
torch.manual_seed(SEED)
residual = ResidualModel()
residual_result = train_open_loop(
    residual,
    train_logs,
    val_logs,
    epochs=EPOCHS,
    lr=LR,
    rollout_every=ROLLOUT_EVERY,
    residual=True,
    patience=PATIENCE,
    name="residual",
)
save_checkpoint("residual", residual, residual_result, residual=True)


residual  5 s windows  51 steps  train 11267  validation 3489  batch 64
residual  epoch 00  5s mse 4.106096e-05  val 3.571938e-05  validation position RMSE 0.1002 m  saved
residual  epoch 01  5s mse 2.634957e-05  val 2.222761e-05
residual  epoch 02  5s mse 2.389142e-05  val 2.162339e-05
residual  epoch 03  5s mse 2.231617e-05  val 2.193739e-05
residual  epoch 04  5s mse 2.084739e-05  val 2.063703e-05
residual  epoch 05  5s mse 1.974927e-05  val 1.974540e-05  validation position RMSE 0.0704 m  saved
residual  epoch 06  5s mse 1.908598e-05  val 1.939332e-05
residual  epoch 07  5s mse 1.863053e-05  val 1.921129e-05
residual  epoch 08  5s mse 1.838091e-05  val 1.811627e-05
residual  epoch 09  5s mse 1.814450e-05  val 1.857755e-05
residual  epoch 10  5s mse 1.810805e-05  val 1.816313e-05  validation position RMSE 0.0652 m  saved
residual  epoch 11  5s mse 1.796189e-05  val 1.784803e-05
residual  epoch 12  5s mse 1.788007e-05  val 1.815530e-05
residual  epoch 13  5s mse 1.788031e-05  val 1.8

In [4]:
residual, residual_result, residual_flag = load_checkpoint("residual")
plot_loss(residual_result, color=COLORS["residual"], label="residual")


In [5]:
residual, _residual_result, residual_flag = load_checkpoint("residual")
plot_rollouts(
    residual,
    val_logs,
    residual=residual_flag,
    color=COLORS["residual"],
    label="residual",
    set_label="Validation",
)


In [6]:
residual, _residual_result, residual_flag = load_checkpoint("residual")
plot_rollouts(
    residual,
    test_logs,
    residual=residual_flag,
    color=COLORS["residual"],
    label="residual",
    set_label="Test",
)


In [7]:
residual, _residual_result, residual_flag = load_checkpoint("residual")
plot_response(residual, train_logs + val_logs, residual=residual_flag)
plot_asymmetry(residual, train_logs + val_logs, label="residual")


### Learning Approaches

We experiment with a series of neural networks that seek to improve upon the unicycle model, and to match the residual baseline above, without being handed that physics prior. We examine five regimes:

1. Naive learning
2. Canonicalization
3. Symmetry across the longitudinal axis
4. Symmetry across both mirrors (left–right and front–back)
5. An even-odd split: two half-size networks whose sum is a canonical map

Each regime changes what the network is allowed to see, not the trunk that fits it. The residual above is the control: a network handed the nominal unicycle. Every learned map, including naive, remixes the scaled ticks into $(\sigma, \delta)$, which are $v$ and $\omega$ up to the wheel radius and the track. Those two lengths stay out of the network. Canonicalization drops the pose and stops there: ungated outputs on that basis. The symmetric maps keep the same coordinates and add mirror constraints on top. That is the fair comparison. Anyone fitting this platform would at least use $(\sigma, \delta)$; the extra assumption is the symmetry.

Every regime uses the same split. Naive, canonical, both symmetric maps, and the residual use the same MLP. The even-odd split is two trunks of about half that width so the total parameter count matches canonical. A run may last 100 epochs. It stops after 10 epochs in a row with no drop in the validation 5 s loss. The saved checkpoint is still the lowest validation open-loop position RMSE, and the loss charts stay on a 0–100 epoch axis so runs can be compared.

#### Training split

Robot 11 only. The test collections are figure 8, diagonal lawnmower reverse, and random trajectory, held out whole. The other eight are coarsened to windows of three ticks (about 0.1 s) before the cut, and a window never crosses the cut. The reverse spiral has encoder dropouts; it stays in because those frames are part of the record.

The first 75% of those eight collections train. The last 25% is validation. The curves below mark that set as validation. The saved checkpoint is the lowest validation open-loop position RMSE.

#### Model

Naive, canonical, both symmetric maps, and the residual use the MLP in `unicycle/models.py`: three ReLU layers of width 32, a width-16 layer, and a linear head. The even-odd split uses two copies of a thinner trunk (hidden 22, encoding 12) so the two together match that parameter count. Every model divides the raw left and right encoder increments by the full-scale tick count on one wheel and remixes them into $(\sigma, \delta) = (\tilde c_L + \tilde c_R,\ \tilde c_R - \tilde c_L)$. Those are $v$ and $\omega$ up to the wheel radius and the track, which never enter the network. Naive concatenates the arena pose. Canonical and the residual stop at $(\sigma, \delta)$ with ungated outputs. Symmetric maps then square and gate those coordinates. The head is initialized at zero and multiplied by one full-scale step, so epoch 0 predicts no motion. There is no recurrent state. On a unicycle the current increments are enough; a quadcopter, whose dynamics remember earlier state, is where a recurrent net starts to matter.

Every model emits the body-frame step over that window,

$$
\begin{pmatrix} \Delta x \\ \Delta y \\ \Delta \theta \end{pmatrix}
=
\operatorname{Log}(T_k^{-1} T_{k+1}),
$$

the discrete counterpart of $(v_x^{\text{body}}, v_y^{\text{body}}, \omega_\theta^{\text{body}})$. Rollout applies the $SE(2)$ exponential and composes it on the current pose. To first order that is

$$
\begin{aligned}
\begin{pmatrix} x_{k+1}^{\text{nav}} \\ y_{k+1}^{\text{nav}} \end{pmatrix}
&=
\begin{pmatrix} x_k^{\text{nav}} \\ y_k^{\text{nav}} \end{pmatrix}
+ R(\theta_k^{\text{nav}})
\begin{pmatrix} \Delta x \\ \Delta y \end{pmatrix} \\
\theta_{k+1}^{\text{nav}} &= \theta_k^{\text{nav}} + \Delta \theta.
\end{aligned}
$$


### Naive Learning Implementation

In the naive case, the network's input is the full $SE(2)$ state and the unicycle coordinates of the ticks:

$$
\left(x^{\text{nav}},\ y^{\text{nav}},\ \theta^{\text{nav}},\ \sigma,\ \delta \right)
$$

with $\sigma = \tilde c_{\text{left}} + \tilde c_{\text{right}}$ and $\delta = \tilde c_{\text{right}} - \tilde c_{\text{left}}$ after dividing by the full-scale count. Those are $v$ and $\omega$ up to the wheel radius and the track. The unicycle model would have turned the ticks into

$$
v = \frac{\Delta c_{\text{right}} + \Delta c_{\text{left}}}{2\,\Delta t}\, r, \qquad
\omega = \frac{\Delta c_{\text{right}} - \Delta c_{\text{left}}}{b\,\Delta t}\, r
$$

with $r$ the meters per tick and $b$ the track width. Naive stops at $(\sigma, \delta)$ and concatenates the pose. The radius and the track never enter. The network still has to learn the rest of that map, and it has to learn it separately at every pose.

**This approach disregards invariance under group transformations; the function does not exploit the symmetry of the body dynamics under $SE(2)$ group actions. Additionally no symmetry is enforced in the architecture or loss**, making this the most unconstrained and general setting. Such training regimes are more susceptible to overfitting, require larger datasets, and typically train more slowly.


In [ ]:
torch.manual_seed(SEED)
naive = NaiveModel()
naive_result = train_open_loop(
    naive,
    train_logs,
    val_logs,
    epochs=EPOCHS,
    lr=LR,
    rollout_every=ROLLOUT_EVERY,
    patience=PATIENCE,
    name="naive",
)
save_checkpoint("naive", naive, naive_result, residual=False)


In [8]:
naive, naive_result, _residual = load_checkpoint("naive")
plot_loss(naive_result, color=COLORS["naive"], label="naive")


In [ ]:
naive, _naive_result, _residual = load_checkpoint("naive")
plot_rollouts(
    naive, val_logs, residual=False, color=COLORS["naive"], label="naive", set_label="Validation"
)


In [ ]:
naive, _naive_result, _residual = load_checkpoint("naive")
plot_rollouts(
    naive, test_logs, residual=False, color=COLORS["naive"], label="naive", set_label="Test"
)


In [ ]:
naive, _naive_result, _residual = load_checkpoint("naive")
plot_response(naive, train_logs + val_logs, residual=False, at_origin=True)


**Takeaways:** The naive model performs terribly.  It is slow to train, unstable, and starts overfitting almost right away.  It underperforms the unicycle model on every trajectory, and is flat out nonsensical on many.  This isn't too suprising.  We're feeding the network x,y, and yaw along with wheel encoders.  Since the body dynamics are mostly indepedent of these, the model is free to use them as an index to memorize twists.  

In essence the model is memorizing a mapping between (x,y,yaw) pairs and twists which is easier than learning the true dynamics.  Given a way a convient way to cheat, the model converges to local minima quickly.  See below to see the strong correlation between network predictions and xy position. 

INSERT THE CODE FOR THIS VISUAL.


### Canonicalization

The body-frame dynamics do not depend on where the robot is in the arena. If $g \in SE(2)$ acts on a pose, the body step is unchanged:

$$
f_{\text{body}}(g \cdot x,\ \Delta c_{\text{left}},\ \Delta c_{\text{right}})
=
f_{\text{body}}(x,\ \Delta c_{\text{left}},\ \Delta c_{\text{right}}).
$$

The unicycle picture of that fact is the twist at the identity,

$$
\begin{pmatrix} v \\ 0 \\ \omega \end{pmatrix},
$$

pushed to the current heading by $g$. The network does not need the pose. It remixes the scaled ticks into

$$
\sigma = \tilde c_{\text{left}} + \tilde c_{\text{right}}, \qquad
\delta = \tilde c_{\text{right}} - \tilde c_{\text{left}},
$$

which are $v$ and $\omega$ up to the wheel radius and the track. Those two lengths stay out of the network. The trunk sees $(\sigma, \delta)$ and the three outputs are ungated: nothing is squared, nothing is multiplied by $\delta$. That is the fair body-frame baseline. The sections after this keep the same coordinates and add mirror constraints on top.


In [ ]:
torch.manual_seed(SEED)
canonical = CanonicalModel()
canonical_result = train_open_loop(
    canonical,
    train_logs,
    val_logs,
    epochs=EPOCHS,
    lr=LR,
    rollout_every=ROLLOUT_EVERY,
    patience=PATIENCE,
    name="canonical",
)
save_checkpoint("canonical", canonical, canonical_result, residual=False)


In [9]:
naive, naive_result, _residual = load_checkpoint("naive")
canonical, canonical_result, _residual = load_checkpoint("canonical")
plot_loss(
    canonical_result,
    color=COLORS["canonical"],
    label="canonical",
    others=[(naive_result, COLORS["naive"], "naive")],
)


In [10]:
canonical, _canonical_result, _residual = load_checkpoint("canonical")
plot_rollouts(
    canonical,
    val_logs,
    residual=False,
    color=COLORS["canonical"],
    label="canonical",
    set_label="Validation",
)


In [11]:
canonical, _canonical_result, _residual = load_checkpoint("canonical")
plot_rollouts(
    canonical,
    test_logs,
    residual=False,
    color=COLORS["canonical"],
    label="canonical",
    set_label="Test",
)


In [12]:
canonical, _canonical_result, _residual = load_checkpoint("canonical")
plot_response(canonical, train_logs + val_logs, residual=False)
plot_asymmetry(canonical, train_logs + val_logs, label="canonical")


**Takeaways**: The cannonical model is decidedely better than the naive model.  It converges more quickly, achieves far superior MAEs, doesn't display the same overfitting behavior, and produces sane trajectories.  However it still has some serious flaws. The model greatly improves results on the sprial collection in comparison to the uniycycle but sees degraded performance on every other collection. Since MSE is squared error large outliers are punished, and the model is encouraged to learn parameters to improve on the spiral trajectory. 

The models outputs are relatively symmettric which is encouraging, but there is still more asymettry than we'd expect from the robotarium unicycles which are very close to symettric.  In theory the asymettry could correspond to true biases in the system, but the degraded performance on most collections suggests that the asymettric parts are likely fitting to dataset biases/noise.

**With a longer training roll out sequence (using 10 second rollout loss instead of 5.0 second, we might stabilize this more), a different loss function, or early stopping, we might see improvement but the point stands: the model is promising, but requires more data, and we have to be concious about the data mixture we give it to prevent overfitting to one trajectory type.**


### Canonicalization and Symmetry Across the Longitudinal Axis

Canonicalization already drops the pose and remixes the ticks into $(\sigma, \delta)$ with an ungated head. Here we keep those coordinates and enforce a mirror across the robot's longitudinal axis.

This is a modeling assumption. The center of mass and the hardware are not perfectly symmetric, so the constraint introduces some error. The claim is that the constraint still wins: faster convergence, less overfitting, and a better held-out path.

Swapping the wheels is that mirror. It leaves the forward step alone and flips the lateral and yaw steps:

$$
f(\Delta c_{\text{right}},\ \Delta c_{\text{left}})
=
\begin{pmatrix} \Delta x \\ -\Delta y \\ -\Delta \theta \end{pmatrix}.
$$

That is the same statement as $f(v, -\omega) = (\Delta x, -\Delta y, -\Delta \theta)$, because a wheel swap flips $\omega$ and leaves $v$ alone. Canonical formed $(\sigma, \delta)$ and stopped there. This section squares $\delta$ and multiplies the lateral and yaw outputs by $\delta$.

Scale the two tick channels by the full-scale count and write

$$
\sigma = \tilde c_{\text{left}} + \tilde c_{\text{right}}, \qquad
\delta = \tilde c_{\text{right}} - \tilde c_{\text{left}}.
$$

$\sigma$ is even under a wheel swap and $\delta$ is odd. The trunk sees

$$
(\sigma,\ \delta^2),
$$

which is unchanged when the wheels swap. It emits $(a, b, c)$, and the step is

$$
\Delta x = a, \qquad \Delta y = \delta \, b, \qquad \Delta \theta = \delta \, c,
$$

then multiplied by the full-scale step. Squaring $\delta$ makes that input even. Multiplying the lateral and yaw channels by $\delta$ itself, rather than by $\operatorname{sign}(\delta)$ or by $|\delta|$, makes those channels odd and forces them through zero when both wheels report the same increment.


In [ ]:
torch.manual_seed(SEED)
symmetric = SymmetricModel()
symmetric_result = train_open_loop(
    symmetric,
    train_logs,
    val_logs,
    epochs=EPOCHS,
    lr=LR,
    rollout_every=ROLLOUT_EVERY,
    patience=PATIENCE,
    name="symmetric",
)
save_checkpoint("symmetric", symmetric, symmetric_result, residual=False)


In [13]:
naive, naive_result, _residual = load_checkpoint("naive")
canonical, canonical_result, _residual = load_checkpoint("canonical")
symmetric, symmetric_result, _residual = load_checkpoint("symmetric")
plot_loss(
    symmetric_result,
    color=COLORS["symmetric"],
    label="symmetric",
    others=[
        (naive_result, COLORS["naive"], "naive"),
        (canonical_result, COLORS["canonical"], "canonical"),
    ],
)


In [ ]:
symmetric, _symmetric_result, _residual = load_checkpoint("symmetric")
plot_rollouts(
    symmetric,
    val_logs,
    residual=False,
    color=COLORS["symmetric"],
    label="symmetric",
    set_label="Validation",
)


In [ ]:
symmetric, _symmetric_result, _residual = load_checkpoint("symmetric")
plot_rollouts(
    symmetric,
    test_logs,
    residual=False,
    color=COLORS["symmetric"],
    label="symmetric",
    set_label="Test",
)


In [ ]:
symmetric, _symmetric_result, _residual = load_checkpoint("symmetric")
plot_response(symmetric, train_logs + val_logs, residual=False)


**Takeaways:** The model training process is miles ahead of the naive and cannonical models.  Training convereges 2x-3x as fast, the 


### Canonicalization and Symmetry Across Both Axes

This model keeps the left–right wheel-swap mirror and adds a second one, across the latitudinal axis: a geometric reflection through the body $yz$-plane, $x\mapsto -x$. The twist is defined about the axle center; sensor offsets are removed by a fixed frame transform.

Write $x=(\Delta c_{\text{left}},\,\Delta c_{\text{right}})$ and $f(x)=(\Delta x,\,\Delta y,\,\Delta\theta)$. The two mirrors are

$$
\begin{aligned}
f(\Delta c_{\text{right}},\,\Delta c_{\text{left}})
&= \begin{pmatrix}\Delta x\\ -\Delta y\\ -\Delta\theta\end{pmatrix},
\\
f(-\Delta c_{\text{left}},\,-\Delta c_{\text{right}})
&= \begin{pmatrix}-\Delta x\\ +\Delta y\\ -\Delta\theta\end{pmatrix}.
\end{aligned}
$$

The first is a swap of the wheels: $v$ stays, $\omega$ flips. The second reverses both wheels. Lateral velocity sits in the mirror plane, so $\Delta y$ is unchanged; $\Delta x$ and the pseudovector $\omega$ reverse. Canonical already formed $(\sigma, \delta)$. This section squares both and gates the outputs.

Scale the ticks as in the last section,

$$
\sigma = \tilde c_{\text{left}} + \tilde c_{\text{right}}, \qquad
\delta = \tilde c_{\text{right}} - \tilde c_{\text{left}}.
$$

Under a wheel swap, $\sigma$ is even and $\delta$ is odd, which is why the last section squared $\delta$. Under the latitudinal mirror both ticks negate, so $\sigma$ and $\delta$ both flip. Squaring $\sigma$ is the same trick on that second odd channel. The trunk sees the pair that is even under both mirrors,

$$
(\sigma^2,\ \delta^2),
$$

which is unchanged when the wheels swap or when both reverse. It emits $(a, b, c)$, and the step is

$$
\Delta x = \sigma\, a, \qquad \Delta y = \sigma\delta\, b, \qquad \Delta\theta = \delta\, c,
$$

then multiplied by the full-scale step (from the actuator limits). Multiplying by $\sigma$ or $\delta$ themselves, rather than by a sign or an absolute value, puts the right parity on each output and forces that channel through zero when the corresponding invariant vanishes.

Check each channel. $\Delta x=\sigma\,a$: a wheel swap leaves $\sigma$ alone, so $\Delta x$ is unchanged; reversing both wheels sends $\sigma\mapsto-\sigma$, so $\Delta x$ flips. $\Delta y=\sigma\delta\,b$: a wheel swap leaves $\sigma$ and flips $\delta$, so $\Delta y$ flips. Reversing both wheels flips **both** $\sigma$ and $\delta$, and the two minus signs cancel, $\sigma\delta\mapsto(+\sigma\delta)$, so $\Delta y$ is unchanged. That is the slip term, $\propto v\omega$, and with the left–right mirror it also gives $v_y(-v,\omega)=-v_y(v,\omega)$. $\Delta\theta=\delta\,c$ flips under both: that is $\omega$. The ideal unicycle is constant $a$ and $c$ with $b=0$. $\Delta y$ is zero when driving straight ($\delta=0$) and when spinning in place ($\sigma=0$), both of which are physically right for slip.


In [ ]:
torch.manual_seed(SEED)
front_back = FrontBackSymmetricModel()
front_back_result = train_open_loop(
    front_back,
    train_logs,
    val_logs,
    epochs=EPOCHS,
    lr=LR,
    rollout_every=ROLLOUT_EVERY,
    patience=PATIENCE,
    name="front_back",
)
save_checkpoint("front_back", front_back, front_back_result, residual=False)


In [14]:
symmetric, symmetric_result, _residual = load_checkpoint("symmetric")
front_back, front_back_result, _residual = load_checkpoint("front_back")
plot_loss(
    front_back_result,
    color=COLORS["front_back"],
    label="two-axis symmetric",
    others=[
        (symmetric_result, COLORS["symmetric"], "symmetric"),
    ],
)


In [ ]:
front_back, _front_back_result, _residual = load_checkpoint("front_back")
plot_rollouts(
    front_back,
    val_logs,
    residual=False,
    color=COLORS["front_back"],
    label="two-axis symmetric",
    set_label="Validation",
)


In [ ]:
front_back, _front_back_result, _residual = load_checkpoint("front_back")
plot_rollouts(
    front_back,
    test_logs,
    residual=False,
    color=COLORS["front_back"],
    label="two-axis symmetric",
    set_label="Test",
)


In [ ]:
front_back, _front_back_result, _residual = load_checkpoint("front_back")
plot_response(front_back, train_logs + val_logs, residual=False)


### Even-Odd Split of the Canonical Map

Any body-step map on $(\sigma, \delta)$ splits uniquely into a piece that obeys the left-right wheel-swap mirror and a piece that does the opposite. Their sum is unconstrained, so this is the same function class as canonical, written as two networks.

Both trunks still see $(\sigma,\,\delta^2)$. The **even** branch is the longitudinal model: wheel swap sends $(\Delta x,\Delta y,\Delta\theta)\mapsto(\Delta x,-\Delta y,-\Delta\theta)$,

$$
\Delta x = a,\qquad \Delta y = \delta\, b,\qquad \Delta\theta = \delta\, c.
$$

The **odd** branch is the complementary identities: wheel swap sends $(\Delta x,\Delta y,\Delta\theta)\mapsto(-\Delta x,\Delta y,\Delta\theta)$,

$$
\Delta x = \delta\, a,\qquad \Delta y = b,\qquad \Delta\theta = c.
$$

Each branch is its own MLP, about half the width of the canonical net (hidden 22, encoding 12), so the two together match the canonical parameter count. Both heads start at zero, so epoch 0 is standing still, the same as canonical. The step is the sum of the two branches.

The odd branch is the leftover after a wheel-swap mirror, so it is the part that can fit a left–right bias. Training adds `ODD_PENALTY` times the mean squared size of that leftover, with yaw written as arc length at half the robot diameter. The even branch is unpenalized. Set `ODD_PENALTY` in the setup cell; 0 drops the extra term.

The checkpoint is written to `even_odd`.


In [ ]:
torch.manual_seed(SEED)
ODD_PENALTY = 1.0
even_odd = EvenOddModel(penalty=ODD_PENALTY)
even_odd_result = train_open_loop(
    even_odd,
    train_logs,
    val_logs,
    epochs=EPOCHS,
    lr=LR,
    rollout_every=ROLLOUT_EVERY,
    patience=PATIENCE,
    name="even_odd",
)
save_checkpoint("even_odd", even_odd, even_odd_result, residual=False)


In [15]:
canonical, canonical_result, _residual = load_checkpoint("canonical")
symmetric, symmetric_result, _residual = load_checkpoint("symmetric")
front_back, front_back_result, _residual = load_checkpoint("front_back")
even_odd, even_odd_result, _residual = load_checkpoint("even_odd")
plot_loss(
    even_odd_result,
    color=COLORS["even_odd"],
    label="even-odd",
    others=[
        (canonical_result, COLORS["canonical"], "canonical"),
        (symmetric_result, COLORS["symmetric"], "symmetric"),
        (front_back_result, COLORS["front_back"], "two-axis"),
    ],
)


In [ ]:
even_odd, _result, _residual = load_checkpoint("even_odd")
plot_rollouts(
    even_odd,
    val_logs,
    residual=False,
    color=COLORS["even_odd"],
    label="even-odd",
    set_label="Validation",
)


In [16]:
even_odd, _result, _residual = load_checkpoint("even_odd")
plot_rollouts(
    even_odd,
    test_logs,
    residual=False,
    color=COLORS["even_odd"],
    label="even-odd",
    set_label="Test",
)


In [17]:
even_odd, _result, _residual = load_checkpoint("even_odd")
plot_response(even_odd, train_logs + val_logs, residual=False)
plot_asymmetry(even_odd, train_logs + val_logs, label="even-odd")


### Summary and Takeaways 

TBD


In [18]:
loaded = {}
for name in ("naive", "canonical", "symmetric", "front_back", "even_odd", "residual"):
    model, result, residual = load_checkpoint(name)
    loaded[name] = (model, result, residual)
regimes = [
    ("naive", *loaded["naive"][:2], False, COLORS["naive"]),
    ("canonical", *loaded["canonical"][:2], False, COLORS["canonical"]),
    ("symmetric", *loaded["symmetric"][:2], False, COLORS["symmetric"]),
    ("two-axis symmetric", *loaded["front_back"][:2], False, COLORS["front_back"]),
    ("even-odd", *loaded["even_odd"][:2], False, COLORS["even_odd"]),
    ("residual", *loaded["residual"][:2], loaded["residual"][2], COLORS["residual"]),
]
print(f"{'model':12s} {'best epoch':>12s} {'validation RMSE':>16s}")
for label, _model, result, _residual, _color in regimes:
    print(f"{label:12s} {result.best_epoch:12d} {result.best_error:15.3f} m")
plot_summary(regimes, val_logs)


model          best epoch  validation RMSE
naive                  85           0.260 m
canonical              76           0.086 m
symmetric              35           0.062 m
two-axis symmetric           45           0.077 m
even-odd               40           0.062 m
residual               15           0.057 m


In [19]:
loaded = {}
for name in ("naive", "canonical", "symmetric", "front_back", "even_odd", "residual"):
    model, result, residual = load_checkpoint(name)
    loaded[name] = (model, result, residual)
regimes = [
    ("naive", *loaded["naive"][:2], False, COLORS["naive"]),
    ("canonical", *loaded["canonical"][:2], False, COLORS["canonical"]),
    ("symmetric", *loaded["symmetric"][:2], False, COLORS["symmetric"]),
    ("two-axis symmetric", *loaded["front_back"][:2], False, COLORS["front_back"]),
    ("even-odd", *loaded["even_odd"][:2], False, COLORS["even_odd"]),
    ("residual", *loaded["residual"][:2], loaded["residual"][2], COLORS["residual"]),
]
plot_summary_paths(regimes, val_logs, set_label="Validation")


In [20]:
loaded = {}
for name in ("naive", "canonical", "symmetric", "front_back", "even_odd", "residual"):
    model, result, residual = load_checkpoint(name)
    loaded[name] = (model, result, residual)
regimes = [
    ("naive", *loaded["naive"][:2], False, COLORS["naive"]),
    ("canonical", *loaded["canonical"][:2], False, COLORS["canonical"]),
    ("symmetric", *loaded["symmetric"][:2], False, COLORS["symmetric"]),
    ("two-axis symmetric", *loaded["front_back"][:2], False, COLORS["front_back"]),
    ("even-odd", *loaded["even_odd"][:2], False, COLORS["even_odd"]),
    ("residual", *loaded["residual"][:2], loaded["residual"][2], COLORS["residual"]),
]
plot_summary_paths(regimes, test_logs, set_label="Test")


### Final Test Metrics

For each whole test collection, report two complementary position errors:

- **Full rollout MSE:** one open-loop rollout from the collection's first Vicon pose, averaged over the complete collection.
- **5.0 s rollout MSE:** restart at every possible Vicon pose, roll open loop for approximately 5 seconds, then average over every predicted pose in every window.

Both are mean squared Euclidean $xy$ error, $\mathbb{E}[\|\hat p-p\|_2^2]$, in m². These test metrics do not choose checkpoints and do not use the training loss mask.

The bars after the table are position RMSE in meters, the same metric as the validation RMSE panel. Each bar pools the three test collections.


In [21]:
test_regimes = [("unicycle", None, False, UNICYCLE)]
for name, label, color in (
    ("naive", "naive", COLORS["naive"]),
    ("canonical", "canonical", COLORS["canonical"]),
    ("symmetric", "symmetric", COLORS["symmetric"]),
    ("front_back", "two-axis symmetric", COLORS["front_back"]),
    ("even_odd", "even-odd", COLORS["even_odd"]),
    ("residual", "residual", COLORS["residual"]),
):
    model, _result, residual = load_checkpoint(name)
    test_regimes.append((label, model, residual, color))

full_by_model = {label: [] for label, _model, _residual, _color in test_regimes}
short_by_model = {label: [] for label, _model, _residual, _color in test_regimes}
print("| test collection | model | full rollout MSE (m²) | 5.0 s rollout MSE (m²) |")
print("|---|---:|---:|---:|")
for log in test_logs:
    collection = log.name.replace("_", " ").replace("lawnmower ", "")
    for label, model, residual, _color in test_regimes:
        full_mse = rollout_position_mse(model, log, residual=residual)
        short_mse = rollout_position_mse(model, log, residual=residual, horizon_s=5.0)
        full_by_model[label].append(full_mse)
        short_by_model[label].append(short_mse)
        print(f"| {collection} | {label} | {full_mse:.6f} | {short_mse:.6f} |")

names = [label for label, _model, _residual, _color in test_regimes]
colors = [color for _label, _model, _residual, color in test_regimes]
full_weight = [log.n_poses for log in test_logs]
short_weight = []
for log in test_logs:
    width = int(round(5.0 / log.dt))
    if width % 2 == 0:
        width += 1
    steps = int(log.increments.shape[0])
    short_weight.append((steps - width + 1) * width)

def _pooled_rmse(mses, weights):
    return (sum(mse * weight for mse, weight in zip(mses, weights)) / sum(weights)) ** 0.5

full_rmse = [_pooled_rmse(full_by_model[label], full_weight) for label in names]
short_rmse = [_pooled_rmse(short_by_model[label], short_weight) for label in names]
print()
print("| model | full RMSE (m) | 5.0 s RMSE (m) |")
print("|---|---:|---:|")
for label, full, short in zip(names, full_rmse, short_rmse):
    print(f"| {label} | {full:.3f} | {short:.3f} |")

plot_test_rmse(names, full_rmse, short_rmse, colors)


| test collection | model | full rollout MSE (m²) | 5.0 s rollout MSE (m²) |
|---|---:|---:|---:|
| figure8 | unicycle | 0.388005 | 0.000109 |
| figure8 | naive | 6.128761 | 0.000825 |
| figure8 | canonical | 0.106623 | 0.000102 |
| figure8 | symmetric | 0.036829 | 0.000050 |
| figure8 | two-axis symmetric | 0.008386 | 0.000047 |
| figure8 | even-odd | 0.071068 | 0.000056 |
| figure8 | residual | 0.229376 | 0.000079 |
| diagonal reverse | unicycle | 0.035883 | 0.000033 |
| diagonal reverse | naive | 3.796800 | 0.000789 |
| diagonal reverse | canonical | 0.108506 | 0.000044 |
| diagonal reverse | symmetric | 0.058570 | 0.000040 |
| diagonal reverse | two-axis symmetric | 0.044604 | 0.000037 |
| diagonal reverse | even-odd | 0.056675 | 0.000036 |
| diagonal reverse | residual | 0.033543 | 0.000030 |
| random trajectory | unicycle | 0.015146 | 0.000074 |
| random trajectory | naive | 0.444829 | 0.000575 |
| random trajectory | canonical | 0.138295 | 0.000084 |
| random trajectory | symmet

### Data efficiency

This cut shortens every training collection instead of dropping whole maneuvers. `TRAIN_FRACTIONS` is the knob, default `0.75, 0.55, 0.35`: the first 75%, 55%, then 35% of each of the eight training logs. `0.75` is the official split. A smaller prefix cannot pass the validation cut, and the unused middle of each log is left out, so the maneuver mix stays the same and only the length changes.

The last quarter of all eight stays validation: it chooses checkpoints and early stopping, and it does not change when the prefix shortens. The three test collections never enter training or checkpoint selection.

A shorter prefix makes each epoch fewer updates. The epoch count is raised in proportion to the 5-second training windows that were cut, so each prefix gets about as many updates as the original 75% run. After each run, the restored checkpoint is scored by open-loop position RMSE on the held-out test set (figure 8, diagonal reverse, random trajectory). Each prefix is trained five times, seeds `SEED` through `SEED+4`. The curve is the mean of those five test RMSEs; the bars are one sample standard deviation.

We don't even include the naive model since it performs so poorly it throws off the scale.  The naive model performed horribly even with all the training data at it's disposal.


In [22]:
import statistics

from unicycle.data import load_efficiency_splits
from unicycle.figures import plot_data_efficiency
from unicycle.rollout import held_out_rmse
from unicycle.train import epochs_for_same_budget, train_open_loop

REPS = 5
TRAIN_FRACTIONS = [0.75, 0.55, 0.35, 0.15]
validation, splits, full_logs = load_efficiency_splits(TRAIN_FRACTIONS)
builders = [
    ("canonical", CanonicalModel, False, COLORS["canonical"]),
    ("symmetric", SymmetricModel, False, COLORS["symmetric"]),
    ("two-axis symmetric", FrontBackSymmetricModel, False, COLORS["front_back"]),
    ("residual", ResidualModel, True, COLORS["residual"]),
]
budget = [
    epochs_for_same_budget(80, full_logs, train_logs)
    for _fraction, train_logs in splits
]
for (fraction, train_logs), epochs in zip(splits, budget):
    print(f"prefix {100 * fraction:.0f}%  {epochs} epochs  {len(train_logs)} collections")

scores = {name: [[] for _ in splits] for name, _builder, _residual, _color in builders}

for split_index, ((fraction, train_logs), epochs) in enumerate(zip(splits, budget)):
    print(f"\nprefix {100 * fraction:.0f}%")
    for name, builder, residual, _color in builders:
        for rep in range(REPS):
            torch.manual_seed(SEED + rep)
            model = builder()
            train_open_loop(
                model,
                train_logs,
                validation,
                epochs=epochs,
                lr=LR,
                rollout_every=ROLLOUT_EVERY,
                residual=residual,
                name=f"{name} {100 * fraction:.0f}% seed {SEED + rep}",
                patience=PATIENCE,
                min_delta=0.0,
            )
            test_rmse = held_out_rmse(model, test_logs, residual=residual)
            scores[name][split_index].append(test_rmse)
            print(
                f"{name:20s}  seed {SEED + rep}  test position RMSE {test_rmse:.4f} m"
            )

means = {
    name: [statistics.fmean(reps) for reps in per_split]
    for name, per_split in scores.items()
}
stds = {
    name: [statistics.stdev(reps) if len(reps) > 1 else 0.0 for reps in per_split]
    for name, per_split in scores.items()
}

header = f"{'prefix':8s} {'epochs':>7s}" + "".join(f"{name:>22s}" for name, _b, _r, _c in builders)
print("\n" + header + "   (mean ± std, 5 seeds)")
for index, ((fraction, _train_logs), epochs) in enumerate(zip(splits, budget)):
    row = f"{100 * fraction:.0f}%".ljust(8) + f"{epochs:7d}"
    for name, _builder, _residual, _color in builders:
        row += f"{means[name][index]:10.4f} ± {stds[name][index]:.4f}".rjust(22)
    print(row)


prefix 75%  80 epochs  8 collections
prefix 55%  111 epochs  8 collections
prefix 35%  179 epochs  8 collections
prefix 15%  467 epochs  8 collections

prefix 75%
canonical 75% seed 0  5 s windows  51 steps  train 11267  validation 3489  batch 64
canonical 75% seed 0  epoch 00  5s mse 6.043180e-02  val 5.037516e-02  validation position RMSE 1.0198 m  saved
canonical 75% seed 0  epoch 01  5s mse 5.747969e-02  val 4.228071e-02
canonical 75% seed 0  epoch 02  5s mse 3.488996e-02  val 1.549826e-02
canonical 75% seed 0  epoch 03  5s mse 1.647175e-02  val 1.214347e-02
canonical 75% seed 0  epoch 04  5s mse 1.495159e-02  val 1.145357e-02
canonical 75% seed 0  epoch 05  5s mse 1.389105e-02  val 1.075216e-02  validation position RMSE 2.0847 m
canonical 75% seed 0  epoch 06  5s mse 1.285517e-02  val 9.912631e-03
canonical 75% seed 0  epoch 07  5s mse 1.155376e-02  val 8.682499e-03
canonical 75% seed 0  epoch 08  5s mse 9.330685e-03  val 6.303663e-03
canonical 75% seed 0  epoch 09  5s mse 5.14518

In [23]:
plot_data_efficiency(
    test_logs,
    [fraction for fraction, _train_logs in splits],
    [(name, color, means[name], stds[name]) for name, _builder, _residual, color in builders],
    budget,
)
